In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from cvxopt import matrix, solvers

from sklearn import datasets
from sklearn import model_selection
from sklearn.datasets import make_circles
import plotly.graph_objects as go
from numba import njit

import neal

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import Classic_SVM_CVXOPT_Gaussian as CG
import Quantum_SVM_Gaussian_neal as QG

import Processing as PC

import os
import json
import time
import pandas as pd

In [7]:
SAVE_DIR_METRIC = r"/home/csb/SVM/Support-Vector-Machine-2/Classic_SVM_DATA/other_data"    # CSV 저장 폴더
SAVE_DIR_ALPHA  = r"/home/csb/SVM/Support-Vector-Machine-2/Classic_SVM_DATA/alpha_data"  # NPZ 저장 폴더

os.makedirs(SAVE_DIR_METRIC, exist_ok=True)
os.makedirs(SAVE_DIR_ALPHA,  exist_ok=True)

In [8]:
# =========================
# Alpha NPZ 저장 경로 생성
# =========================
def build_alpha_path(seed: int, run_id: int, i: int, gamma: float, B: int, K: int, xi: int, C: float) -> str:
    # seed별 폴더로 분리 (원치 않으면 이 2줄 제거하고 바로 SAVE_DIR_ALPHA에 저장하면 됨)
    seed_dir = os.path.join(SAVE_DIR_ALPHA, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    # 파일명 구성
    # (gamma는 소수점 포함 → 파일명 안전하게 포맷)
    return os.path.join(
        seed_dir,
        f"alpha_seed{seed}_run{run_id:03d}_idx{i:02d}_g{gamma:.2f}_B{B}_K{K}_xi{xi}_C{int(C)}.npz"
    )

In [9]:
# =========================
# Main experiment
#   - 기존 코드 유지
#   - α 저장만 추가 (C_alpha, Q_alpha(top-k), Q_energy)
# =========================
def run_single_experiment(N_train, X_train, y_train, X_test, y_test, run_id, split_id, seed):
    rows = []

    B_list = [5]*30
    K_list = [3]*30
    xi_list = [1]*30
    gamma_list = [g/10 for g in range(30)]

    for i in range(len(gamma_list)):
        B = B_list[i]; K = K_list[i]; xi = xi_list[i]; gamma = gamma_list[i]

        # ---- C 계산 (기존 유지)
        C = 0
        for k in range(K):
            C += B**k

        # =======================
        # Classic SVM
        # =======================
        P, q, G, h, A, b, K_train_train = CG.Solver_Parameter(N_train, X_train, y_train, gamma, C)
        sol = CG.Solver_SVM(P, q, G, h, A, b)

        alpha_c = np.array(sol["x"]).reshape(-1)

        C_acc_train, C_auroc_train, C_auprc_train, C_scores_train = CG.evaluate_train(
            y_train, alpha_c, K_train_train, C
        )

        C_acc_test, C_auroc_test, C_auprc_test = CG.evaluate_test(
            y_test,
            CG.Test_evlauation(X_train, X_test, y_train, alpha_c, K_train_train, gamma, C)
        )

        C_gap_acc, C_gap_auroc, C_gap_auprc = CG.Evaluate_Overfitting(
            C_acc_train, C_acc_test, C_auroc_train, C_auroc_test, C_auprc_train, C_auprc_test
        )

        C_loss_train, C_loss_test = CG.Hinge_Loss(
            X_train, X_test, y_train, y_test, alpha_c, K_train_train, C_scores_train, gamma, C
        )
        C_hinge_gap = C_loss_test - C_loss_train

        # ---- Classic primal
        C_J_w, C_J_xi = CG.Primal(alpha_c, K_train_train, y_train, C)

       

        # =======================
        # α 파일 저장 (추가)
        # =======================

        alpha_path = build_alpha_path(seed, run_id, i, gamma, B, K, xi, C)

        np.savez_compressed(
            alpha_path,
            C_alpha=alpha_c.astype(np.float64)
        )

        # =======================
        # CSV row (기존 + alpha_path만 추가)
        # =======================
        rows.append({
            "seed": seed,
            "run": run_id,
            "split": split_id,
            "i": i,
            "B": B, "K": K, "xi": xi, "gamma": gamma, "C": C,

            "C_acc_train": C_acc_train, "C_acc_test": C_acc_test, "C_gap_acc": C_gap_acc,
            "C_auroc_train": C_auroc_train, "C_auroc_test": C_auroc_test, "C_gap_auroc": C_gap_auroc,
            "C_auprc_train": C_auprc_train, "C_auprc_test": C_auprc_test, "C_gap_auprc": C_gap_auprc,
            "C_hinge_gap": C_hinge_gap,

            "C_J_w": C_J_w,
            "C_J_xi": C_J_xi
        })

    return rows

In [10]:
N_RUNS = 1

for seed in range(43, 44):
    all_rows = []
    n_train = 200
    raw = 5
    col = int(n_train / raw)

    X, Y = make_circles(n_samples=500, noise=0.1, random_state=seed)
    X_train, y_train, X_test, y_test = PC.Processing(X, Y, n_train, raw, col)
    kernal_size = len(X_train[0])

    for run_id in range(1, N_RUNS + 1):
        print(f"[SEED {seed}] [RUN {run_id}/{N_RUNS}]")
        for split_id in range(raw):
            all_rows.extend(run_single_experiment(
                kernal_size, 
                X_train[split_id],
                y_train[split_id],
                X_test,
                y_test,
                run_id,
                split_id,
                seed)
                )

    df = pd.DataFrame(all_rows)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"SVM_seed{seed}_runs{N_RUNS}_{timestamp}.csv"
    save_path = os.path.join(SAVE_DIR_METRIC, filename)

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"Saved results to:\n{save_path}")
    print(f"Alpha NPZ saved under:\n{SAVE_DIR_ALPHA}")

[SEED 43] [RUN 1/1]
     pcost       dcost       gap    pres   dres
 0: -6.3840e+02 -3.4224e+03  3e+03  1e-14  1e-14
 1: -8.5641e+02 -1.2432e+03  4e+02  4e-15  1e-14
 2: -1.1748e+03 -1.2160e+03  4e+01  6e-14  4e-14
 3: -1.1777e+03 -1.1786e+03  9e-01  5e-14  3e-14
 4: -1.1780e+03 -1.1780e+03  9e-03  4e-14  4e-14
 5: -1.1780e+03 -1.1780e+03  9e-05  4e-14  4e-14
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -3.0188e+02 -6.4101e+03  6e+03  2e-14  2e-14
 1: -4.5079e+02 -1.0407e+03  6e+02  1e-14  1e-14
 2: -6.1879e+02 -7.7714e+02  2e+02  1e-14  2e-14
 3: -6.6485e+02 -7.1906e+02  5e+01  7e-15  1e-14
 4: -6.8695e+02 -6.9511e+02  8e+00  2e-14  2e-14
 5: -6.9033e+02 -6.9097e+02  6e-01  1e-14  1e-14
 6: -6.9061e+02 -6.9062e+02  1e-02  2e-15  2e-14
 7: -6.9062e+02 -6.9062e+02  1e-04  5e-15  2e-14
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0:  3.4504e+01 -7.0765e+03  7e+03  1e-14  1e-14
 1: -1.9649e+02 -1.3681e+03  1e+03  8e-15  8e-15
 2: 